# INVEN 공식 문서(GUIDE) 데이터 전처리 (CHUNKING)

> - 데이터 파일 : RAG/maple_guide_normalized.json


In [2]:
# [환경 설정] 필요한 라이브러리와 한글 폰트 설정

import re
import json
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from kiwipiepy import Kiwi
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

kiwi = Kiwi()   # 한국어 형태소 분석기(자바 불필요)

In [3]:
# INVEN 공식문서(GUIDE-정규표현식 반영) 가져오기
with open('../data/RAG/maple_guide_normalized.json', encoding='utf-8') as f :
    guide_dict = json.load(f)

# JSON 파일의 DICT -> 판다스 DataFrame 으로 변환
guide_df = pd.DataFrame(guide_dict)

# GUIDE 데이터 확인
display(guide_df.info())
display(guide_df['category'].unique())
display(guide_df.head())



<class 'pandas.core.frame.DataFrame'>
Index: 106 entries, 0 to 105
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   article_id          106 non-null    int64 
 1   title               106 non-null    object
 2   content             106 non-null    object
 3   url                 106 non-null    object
 4   category            106 non-null    object
 5   board_id            106 non-null    int64 
 6   normalized_content  106 non-null    object
dtypes: int64(2), object(5)
memory usage: 6.6+ KB


None

array(['기초 가이드', '성장', '아이템', '사냥/보스 컨텐츠', '스페셜 컨텐츠', '커뮤니티', '거래',
       '캐시 & 코디', '기타/TIP'], dtype=object)

,article_id,title,content,url,category,board_id,normalized_content
0,272,게임 시작,■ 목차\n1. 메이플스토리 계정 만들기\n2. 게임 설치하기\n\n\n\n\n\n...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337,목차 1 메이플스토리 계정 만들기 2 게임 설치하기 메이플스토리 계정 만들기 1 회...
1,373,캐릭터 생성/삭제,■ 목차\n1. 캐릭터 직업 선택하기\n2. 캐릭터 설정하기\n3. 캐릭터 삭제하기...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337,목차 1 캐릭터 직업 선택하기 2 캐릭터 설정하기 3 캐릭터 삭제하기 캐릭터 생성 ...
2,307,캐릭터 이름 변경,■ 목차\n1. 캐릭터 이름 변경 방법\n2. 캐릭터 이름 변경 시 유의사항\n\n...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337,목차 1 캐릭터 이름 변경 방법 2 캐릭터 이름 변경 시 유의사항 캐릭터 이름 변경...
3,417,캐릭터 프리셋,■ 목차\n\n1. 캐릭터 프리셋이란?\n\n2. 캐릭터 프리셋 사용 방법\n\n3...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337,목차 1 캐릭터 프리셋이란 2 캐릭터 프리셋 사용 방법 3 장비 프리셋 사용 방법 ...
4,432,조작법/키세팅,■ 목차\n1. 기본 조작키\n2. 단축키 설정하기\n3. 퀵슬롯 설정하기\n4. ...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337,목차 1 기본 조작키 2 단축키 설정하기 3 퀵슬롯 설정하기 4 스킬 매크로 등록하...
